# Challenge 02 — Inteligencia Geo-Temporal y de Redes

**Optimización de Activos Críticos: TechLogistics S.A.**

EAFIT · Maestría en Ciencia de Datos y Analítica · Periodo 2026-1

Este notebook desarrolla el análisis multidimensional (geoespacial, series de tiempo, procesamiento de señales y grafos) solicitado en el Challenge 03.

Importación de librerías

In [8]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Fase 1: Data Understanding y Geo-Visualización

### Tarea 1: Exploración Geo-Temporal

#### Diccionario de variables de agro_clean.csv

Extraído de Lecture_03_dictionary.pdf

| Variable | Descripción | Naturaleza / Propósito |
|---|---|---|
| `Agro_1` – `Agro_3` | Variables hídricas: Humedad, Evapotranspiración y Humedad Relativa (RH) | Correlación alta entre sí. Estacionarias, **I(0)** |
| `Agro_4` | Radiación PAR (fotosintéticamente activa) | Cíclica — ciclo día/noche |
| `Agro_5` – `Agro_7` | Índices bióticos: **NDVI** y Biomasa | **No estacionarias, I(1)** — tienden a mostrar deriva (drift) en el tiempo |
| `Agro_8` – `Agro_10` | Suelo y viento | Estacionarias — ruido blanco con media constante |
| `Latitude` | Eje Y espacial | Posicionamiento del sensor en el oriente antioqueño |
| `Longitude` | Eje X espacial | Coordenadas decimales, usadas para clustering geoespacial |
| `Source_Node` | ID del sensor (nodo de origen) | Topología de la red mesh del cultivo |
| `Target_Node` | ID del gateway (nodo de destino) | Concentrador de datos al que reporta el sensor |

Carga del dataset y exploración rápida para confirmar tipos de dato, ausencia de nulos y los rangos de `Latitude`/`Longitude`.

In [9]:
agro = pd.read_csv("../data/agro_clean.csv")

print("Dimensiones:", agro.shape)
agro.info()
agro[["Latitude", "Longitude", "Agro_1", "Agro_5"]].describe()

Dimensiones: (2000, 14)
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Agro_1       2000 non-null   float64
 1   Agro_2       2000 non-null   float64
 2   Agro_3       2000 non-null   float64
 3   Agro_4       2000 non-null   float64
 4   Agro_5       2000 non-null   float64
 5   Agro_6       2000 non-null   float64
 6   Agro_7       2000 non-null   float64
 7   Agro_8       2000 non-null   float64
 8   Agro_9       2000 non-null   float64
 9   Agro_10      2000 non-null   float64
 10  Latitude     2000 non-null   float64
 11  Longitude    2000 non-null   float64
 12  Source_Node  2000 non-null   int64  
 13  Target_Node  2000 non-null   int64  
dtypes: float64(12), int64(2)
memory usage: 218.9 KB


,Latitude,Longitude,Agro_1,Agro_5
count,2000.000000,2000.000000,2000.000000,2000.000000
mean,6.202395,-75.397948,60.469678,1.158403
std,0.058344,0.057330,10.519590,0.379320
min,6.100010,-75.499898,43.554852,0.417623
25%,6.153419,-75.447321,49.997291,0.856683
50%,6.204413,-75.397947,61.202955,1.070661
75%,6.252968,-75.349042,70.809795,1.429127
max,6.299980,-75.300241,76.082501,2.321211


In [10]:
agro.head()

,Agro_1,Agro_2,Agro_3,Agro_4,Agro_5,Agro_6,Agro_7,Agro_8,Agro_9,Agro_10,Latitude,Longitude,Source_Node,Target_Node
0,60.248357,47.797447,65.395554,0.000000,0.478718,2.498349,10.000000,6.432151,1.258741,4.663639,6.203680,-75.400366,4,17
1,60.080940,48.076702,66.143237,25.002075,0.467100,2.473166,10.006672,6.469450,1.106051,4.683748,6.282310,-75.474525,14,21
2,60.623974,48.002378,66.342755,49.941596,0.449259,2.464547,10.013349,6.440262,1.183610,8.353141,6.165190,-75.476937,14,22
3,61.211672,48.267738,66.826015,74.756163,0.439299,2.500284,10.020030,6.511042,1.197917,6.213966,6.131334,-75.468152,8,25
4,60.483063,47.912028,65.703353,99.383693,0.436016,2.564177,10.026716,6.619718,1.200795,8.720976,6.241746,-75.349805,5,26


Observamos que la data esta completa y sin atípicos para cada una de las variables. Los rangos los usaremos para el zoom del mapa

Le pedimos a la IA el gráfico con el siguiente prompt:

***Usando plotly express grafica un scatter_mapbox para ver la ubicación de los sensores en el oriente antioqueño. Codifica el color de los puntos según el NVDI usando Agro_5 y el tamaño según la humedad usando Agro_1. Utiliza el rango para definir el zoom adecuado para enfocar el mapa en el oriente antioqueño***

In [11]:
centro_mapa = {"lat": agro["Latitude"].mean(), "lon": agro["Longitude"].mean()}

fig_mapa = px.scatter_mapbox(
    agro,
    lat="Latitude",
    lon="Longitude",
    color="Agro_5",
    size="Agro_1",
    color_continuous_scale="RdYlGn",
    size_max=12,
    zoom=10,
    center=centro_mapa,
    mapbox_style="carto-positron",
    hover_data=["Source_Node", "Target_Node", "Agro_1", "Agro_5"],
    labels={"Agro_5": "NDVI", "Agro_1": "Humedad"},
    title="Sensores agroindustriales — Oriente Antioqueño (color = NDVI, tamaño = Humedad)",
)
fig_mapa.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig_mapa.show()

C:\Users\DanielSantiagoCadavi\AppData\Local\Temp\ipykernel_6716\2944694611.py:3: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_mapa = px.scatter_mapbox(


Al revisar el mapa no se identifica un patrón espacial claro de clustering para el NDVI los puntos con `Agro_5` bajo aparecen dispersos de manera prácticamente uniforme sobre toda el área cubierta por los sensores, sin concentrarse en una subzona particular. Esto se presenta también para todos los niveles de NDVI.

Por tanto, no hay evidencia de clustering geoespacial de biomasa baja. La variabilidad de NDVI que se ve en el mapa puede deberse a otros factores como una deriva temporal (posible estacionalidad o degradación del cultivo) y no a la ubicación geográfica del sensor.

#### Exploración complementaria para la Pregunta de Validación 4: 
¿Cómo influye la posición geográfica en la varianza de la señal
capturada?

Para responder esta pregunta, le hacemos la siguiente solicitud a la IA:

***Divide el área de estudio en 4 cuadrantes usando la mediana de cada coordenada como líneas de corte: Norte/Sur y Este/Oeste. Para cada zona calcula la varianza de `Agro_1` (Humedad) y `Agro_5` (NDVI) y comparalas mediante una tabla y un gráfico de barras, en 2 subplot diferente cada variable***

In [13]:
lat_zona = pd.qcut(agro["Latitude"], 2, labels=["Sur", "Norte"])
lon_zona = pd.qcut(agro["Longitude"], 2, labels=["Oeste", "Este"])
agro["Zona"] = lat_zona.astype(str) + "-" + lon_zona.astype(str)

varianza_por_zona = (
    agro.groupby("Zona")[["Agro_1", "Agro_5"]]
    .var()
    .rename(columns={"Agro_1": "Var(Humedad)", "Agro_5": "Var(NDVI)"})
)

fig_var = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=("Varianza de Humedad (Agro_1)", "Varianza de NDVI (Agro_5)"),
)
fig_var.add_trace(
    go.Bar(
        x=varianza_por_zona.index,
        y=varianza_por_zona["Var(Humedad)"],
        marker_color="#1f77b4",
        name="Var(Humedad)",
    ),
    row=1,
    col=1,
)
fig_var.add_trace(
    go.Bar(
        x=varianza_por_zona.index,
        y=varianza_por_zona["Var(NDVI)"],
        marker_color="#2ca02c",
        name="Var(NDVI)",
    ),
    row=2,
    col=1,
)
fig_var.update_yaxes(title_text="Varianza", row=1, col=1)
fig_var.update_yaxes(title_text="Varianza", row=1, col=2)
fig_var.update_layout(
    title_text="Varianza de Humedad y NDVI por zona geográfica", showlegend=False
)
fig_var.show()

varianza_por_zona

,Var(Humedad),Var(NDVI)
Zona,,
Norte-Este,112.833401,0.137334
Norte-Oeste,111.653211,0.154136
Sur-Este,106.568089,0.148504
Sur-Oeste,111.905746,0.135431


De la gráfica de barras y la tabla podemos ver que:
- La varianza de `Agro_1` (Humedad) varía poco entre zonas (106 a 112, dispersión relativa de 6% aproximadamente)
- La varianza de `Agro_5` (NDVI) muestra una dispersión algo mayor entre zonas en términos porcentuales (0.135 a 0.154, 13% aprox).

En ambos casos las diferencias son pequeñas y ninguna zona se diferencia notablemente de las demás. Esto sugiere que la posición geográfica no tiene influencia sobre la varianza de las señales capturadas: los sensores registran una variabilidad relativamente homogénea sin importar el cuadrante del oriente antioqueño en el que estén ubicados. 

Esto es coherente con el análisis del mapa: en este dataset, la varianza de la señal parece explicarse más por otros factores que por la geografía.

### Tarea 2: Análisis de Estacionariedad y Windowing

## Fase 2: Procesamiento de Señales y Filtrado

### Tarea 3: Análisis Espectral (FFT) y Espectrogramas

### Tarea 4: Filtrado y Reconstrucción

## Fase 3: Análisis de Grafos y Topología de Red

### Tarea 5: Construcción de la Red de Sensores/Subestaciones

## Fase 4: Modelado y Toma de Decisiones (CRISP-DM)

### P1: Causalidad y Redes

### P2: Optimización Geo-Agrónoma

### P3: Analítica Predictiva (ARIMAX)